# Draft Bridge Demo

This notebook shows how to use draft-derived utilities now integrated in `rheed_core.core.draft_bridge`.

## What is covered
- median denoising
- FFT band-pass filtering
- peak detection and cycle segmentation
- per-cycle relaxation time (`tau`) estimation


In [ ]:
import numpy as np

from rheed_core.core.draft_bridge import (
    bandpass_filter_fft,
    detect_peaks_1d,
    estimate_latest_cycle_tau,
    median_filter_1d,
)

In [ ]:
# Build a synthetic RHEED-like cycle signal: repeated exponential recovery with noise.
dt = 0.05
period = 2.0
tau_true = 0.45
t = np.arange(0.0, 30.0, dt)
phase = np.mod(t, period)
rng = np.random.default_rng(4)
y = 1.0 - np.exp(-phase / tau_true) + 0.03 * rng.standard_normal(t.shape[0])

y_med = median_filter_1d(y, kernel_size=5)
y_bp = bandpass_filter_fft(y_med, low_cutoff=0.05, high_cutoff=3.0, sample_frequency=1.0 / dt)

min_distance = int(0.6 * period / dt)
peaks = detect_peaks_1d(y_bp, min_distance=min_distance, prominence=0.05)
tau_est = estimate_latest_cycle_tau(
    t,
    y_bp,
    min_distance=min_distance,
    prominence=0.05,
    mode="growth",
    min_points=8,
)

print(f"Detected peaks: {len(peaks)}")
tau_value = None if tau_est is None else tau_est.tau_s
print(f"Estimated tau: {tau_value}")
print(f"Ground-truth tau: {tau_true:.4f} s")


## Adapting to your data
1. Replace synthetic `t`, `y` with time/intensity arrays from your exported TSST signal.
2. Tune `min_distance` to about `0.6 * expected_period / dt`.
3. Tune `prominence` so obvious oscillation peaks are detected but noise peaks are ignored.
4. Compare estimated `tau` trends against known growth transitions.
